# Electric and gas figures

**Output family:** ELECTRIC AND GAS FIGURES

Builds the per-tract electric and gas operational figures: account counts and energy affordability enrolment.

**What it produces**

- `Data/out/coned_operational_v1_0-2010.json`

## Where this sits in the execution order

Nothing in this package is numbered. Each guide and notebook is named after
its output family, and the order to run them in is this one:

| | Guide | Command | Produces |
|---|---|---|---|
| Step 1 | `docs/tract-shapes.docx, docs/territory-overlays.docx` | `python scripts/update_map_data.py --vintage 2010` | tract shapes AND the territory overlay: one command, two outputs |
| Step 2 | `docs/dac-indicators.docx` | `python scripts/convert_nyserda_raw.py --version 1.0 --geoid-vintage 2010 --raw-date 2023-03-27` | the DAC indicator dataset |
| Step 3 **<- you are here** | `docs/electric-and-gas-figures.docx` | `python scripts/build_coned_dataset.py --vintage 2010` | the electric and gas figures |


## How to use this notebook

Run the cells in order. This notebook is an **orchestrator**: it calls the
package's own scripts and reimplements nothing, so what it produces is
byte-identical to running the same commands in a terminal.

Nothing here contacts the dashboard, Dataverse, or any Con Edison system.
Uploading is a separate manual step, described in the guide.


## Setup: unpack the package, and start the clock

Upload `coned-dac-dashboard-data-tools.zip` to this session first, then
run this cell. If the package is already unpacked it says so and does
nothing.


In [ ]:
# SETUP. Unpacks the package if it is not already unpacked, and starts
# the clock for the timing block at the end.
import datetime, glob, hashlib, json, os, subprocess, sys, time, zipfile

STARTED_AT = datetime.datetime.now(datetime.timezone.utc)
STARTED_CLOCK = time.time()
STAGE_TIMES = []

ZIP_NAME = 'coned-dac-dashboard-data-tools.zip'
PKG_DIR = 'coned-dac-dashboard-data-tools'

def _looks_like_pkg(d):
    return (os.path.isfile(os.path.join(d, 'MANIFEST.txt'))
            and os.path.isdir(os.path.join(d, 'scripts'))
            and os.path.isdir(os.path.join(d, 'Data')))

# Already unpacked anywhere sensible? Then do nothing at all.
_found = [d for d in [os.path.join(os.getcwd(), PKG_DIR),
                      os.path.join('/content', PKG_DIR)]
          if _looks_like_pkg(d)]
_found += [d for d in sorted(glob.glob('/content/**/' + PKG_DIR,
                                       recursive=True))
           if _looks_like_pkg(d)]

if _found:
    print('already unpacked :', _found[0])
    print('nothing to do. Skipping the unpack.')
else:
    _zips = [z for z in ['/content/' + ZIP_NAME,
                         os.path.join(os.getcwd(), ZIP_NAME)]
             if os.path.isfile(z)]
    _zips += sorted(glob.glob('/content/**/' + ZIP_NAME, recursive=True))
    if not _zips:
        raise SystemExit(
            'No package and no zip. Upload ' + ZIP_NAME + ' to this Colab '
            'session (the folder icon in the left sidebar, then the upload '
            'button) and run this cell again.')
    _z = _zips[0]
    _dest = '/content' if os.path.isdir('/content') else os.getcwd()
    print('zip found        :', _z)
    print('unpacking into   :', _dest)
    with zipfile.ZipFile(_z) as _zf:
        _zf.extractall(_dest)
    print('unpacked         :', len(os.listdir(os.path.join(_dest, PKG_DIR))),
          'entries')

print()
print('STARTED', STARTED_AT.strftime('%Y-%m-%d %H:%M:%S UTC'))


## Find the package


In [ ]:
# Find the unpacked package. Nothing here writes anything.
import hashlib, os, subprocess, sys, json, glob

# If you unpacked somewhere this does not find, set it by hand:
#   PKG = '/content/coned-dac-dashboard-data-tools'
PKG = None

def _looks_like_pkg(d):
    return (os.path.isfile(os.path.join(d, 'MANIFEST.txt'))
            and os.path.isdir(os.path.join(d, 'scripts'))
            and os.path.isdir(os.path.join(d, 'Data')))

if PKG is None:
    here = os.getcwd()
    candidates = [here, os.path.dirname(here)]
    candidates += sorted(glob.glob('/content/**/coned-dac-dashboard-data-tools',
                                   recursive=True))
    candidates += sorted(glob.glob(os.path.join(here, '**',
                         'coned-dac-dashboard-data-tools'), recursive=True))
    for c in candidates:
        if c and _looks_like_pkg(c):
            PKG = c
            break

if PKG is None:
    raise SystemExit('Could not find the package root. The setup cell above '
                     'unpacks the zip: run it first. If you unpacked somewhere '
                     'unusual, set PKG at the top of this cell to the folder '
                     'holding MANIFEST.txt, scripts/ and Data/.')

print('package root :', PKG)
print('contents     :', ', '.join(sorted(os.listdir(PKG))))


## Integrity: is this package intact?


In [ ]:
# INTEGRITY. Two halves, and the second is the one that proves something.
#
# 1. The zip's own size and sha256, printed for you to compare against the
#    figures in the handoff note. A notebook INSIDE the zip cannot contain
#    the zip's own digest, so this prints rather than asserts.
# 2. Every unpacked file against the full sha256 in MANIFEST.txt. This is
#    the real check, and it is only possible because MANIFEST.txt carries
#    complete 64-character digests.

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

zips = sorted(glob.glob('/content/**/coned-dac-dashboard-data-tools*.zip',
                        recursive=True))
zips += sorted(glob.glob(os.path.join(os.path.dirname(PKG),
                         'coned-dac-dashboard-data-tools*.zip')))
if zips:
    z = zips[0]
    print('zip          :', z)
    print('  bytes      :', os.path.getsize(z))
    print('  sha256     :', sha256_of(z))
else:
    print('zip          : not found (fine: it may have been deleted after unpacking)')

# MANIFEST.txt rows look like:
#   <path>  <bytes>
#       sha256  <64 hex>
rows, path_now = [], None
with open(os.path.join(PKG, 'MANIFEST.txt'), encoding='utf-8') as fh:
    for line in fh:
        s = line.rstrip('\n')
        t = s.strip()
        if t.startswith('sha256 ') and path_now:
            rows.append((path_now[0], path_now[1], t.split(None, 1)[1].strip()))
            path_now = None
        elif s.startswith(' ') or not t or t.startswith('-') or t.startswith('='):
            continue
        else:
            parts = t.rsplit(None, 1)
            if len(parts) == 2 and parts[1].isdigit():
                path_now = (parts[0].strip(), int(parts[1]))

bad, checked = [], 0
for rel, size, digest in rows:
    full = os.path.join(PKG, rel)
    if not os.path.exists(full):
        bad.append('%s: listed in MANIFEST, absent from the package' % rel)
        continue
    if len(digest) != 64:
        bad.append('%s: MANIFEST digest is %d characters, not 64' % (rel, len(digest)))
        continue
    actual_size, actual = os.path.getsize(full), sha256_of(full)
    if actual_size != size:
        bad.append('%s: %d bytes on disk, MANIFEST says %d' % (rel, actual_size, size))
    if actual != digest:
        bad.append('%s: sha256 mismatch' % rel)
    checked += 1

print()
print('MANIFEST rows parsed   :', len(rows))
print('files verified         :', checked)
if bad:
    print('PROBLEMS               :', len(bad))
    for b in bad:
        print('   ', b)
    raise SystemExit('the package does not match its own MANIFEST; stopping.')
print('every file matches MANIFEST.txt on both size and full sha256.')


## Install the dependencies


In [ ]:
# Dependencies, from the package's own requirements.txt. Nothing pinned
# here by hand: the file in the package is the source of truth.
req = os.path.join(PKG, 'scripts', 'requirements.txt')
print(open(req, encoding='utf-8').read())
p = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', req],
                   capture_output=True, text=True)
print(p.stdout[-2000:])
print(p.stderr[-2000:])
if p.returncode != 0:
    raise SystemExit('pip install failed with %d' % p.returncode)
print('dependencies installed.')


## The run helper


In [ ]:
# The run helper. Every command below goes through this, so the command
# actually issued is visible and a non-zero exit stops the notebook instead
# of scrolling past.
#
# cwd is the PACKAGE ROOT, which is where the guides say to run from: the
# scripts resolve Data/ from their own location, and the paths they print are
# relative to the root.
# It also times every stage, for the timing block at the end.
def run(args, expect=0, stage=None):
    label = stage or ' '.join(args[:1])
    print('$ python ' + ' '.join(args))
    print('-' * 70)
    t0 = time.time()
    p = subprocess.run([sys.executable] + args, cwd=PKG,
                       capture_output=True, text=True)
    elapsed = time.time() - t0
    sys.stdout.write(p.stdout)
    if p.stderr.strip():
        print('--- stderr ---')
        sys.stdout.write(p.stderr)
    print('-' * 70)
    print('exit code: %d      elapsed: %s' % (p.returncode, hms(elapsed)))
    STAGE_TIMES.append((label, elapsed))
    if expect is not None and p.returncode != expect:
        raise SystemExit('expected exit %s, got %d' % (expect, p.returncode))
    return p

def hms(seconds):
    seconds = int(round(seconds))
    return '%d:%02d:%02d' % (seconds // 3600, (seconds % 3600) // 60,
                             seconds % 60)


## Dry run

This script has no dry-run flag, so there is no dry-run cell to offer.
Said plainly rather than faked: the run below is the first thing that
writes. It refuses rather than overwriting an existing output unless you
pass `--force`.


## The real run

Reads the Con Edison electric and gas extracts, and the tract geometry
from step 1 as its tract list.


In [ ]:
run(['scripts/build_coned_dataset.py', '--vintage', '2010'], stage='electric and gas figures, vintage 2010')


## Verify the outputs

Sizes and digests are measured from the files the run just wrote.


In [ ]:
# VERIFICATION. Measure what the run produced, and compare it against
# values that were measured from a real build. A value of None means no
# measurement exists yet: the cell records what it found and says so rather
# than comparing against a number nobody measured.
EXPECTED = [
    ('Data/out/coned_operational_v1_0-2010.json', 176215, 'b9e7a4d6e971b2b3a85f1d135c70f7da317153664c04e3954984b04e6ada3365'),
]

problems = []
for rel, exp_size, exp_sha in EXPECTED:
    full = os.path.join(PKG, rel)
    print(rel)
    if not os.path.exists(full):
        print('    MISSING: the run did not produce this file')
        problems.append('%s is missing' % rel)
        continue
    size, digest = os.path.getsize(full), sha256_of(full)
    print('    bytes  :', size,
          '' if exp_size is None else ('(expected %d)' % exp_size))
    print('    sha256 :', digest)
    if exp_sha is None:
        print('    expected sha256: not measured; recorded, not compared')
    else:
        print('    expected       :', exp_sha)
    if exp_size is not None and size != exp_size:
        problems.append('%s: %d bytes, expected %d' % (rel, size, exp_size))
    if exp_sha is not None and digest != exp_sha:
        problems.append('%s: sha256 differs from the expected build' % rel)


print()
if problems:
    for p_ in problems:
        print('PROBLEM:', p_)
    raise SystemExit('verification failed.')
print('verification passed.')


## How long it took

The block below is the one to screenshot for the run record.


In [ ]:
# TIMING. One block, meant to be read and screenshotted.
FINISHED_AT = datetime.datetime.now(datetime.timezone.utc)
total = time.time() - STARTED_CLOCK

print('=' * 58)
print('RUN TIMING')
print('=' * 58)
for label, secs in STAGE_TIMES:
    print('  %-42s %s' % (label, hms(secs)))
if STAGE_TIMES:
    print('-' * 58)
print('  STARTED        %s' % STARTED_AT.strftime('%Y-%m-%d %H:%M:%S UTC'))
print('  FINISHED       %s' % FINISHED_AT.strftime('%Y-%m-%d %H:%M:%S UTC'))
print('  TOTAL ELAPSED  %s' % hms(total))
print('=' * 58)
print()
print('Times vary by machine. A slower computer taking two or three times')
print('as long is normal. Ten times as long is worth investigating rather')
print('than waiting out: check the preflight table for something being')
print('downloaded that should already be here.')


## What happens next

Nothing in this notebook uploads anything. Follow the guide's upload
section to publish these files through the dashboard's Map Layers card.
